In [1]:
import sys
import os
# Add parent directory to path to import local modules
if '..' not in sys.path:
    sys.path.append('..')

import copy
from datetime import datetime
from einops import rearrange
import torch
from torch.func import vmap
from PIL import Image
import numpy as np
import skdim
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from ncut_pytorch import NCUT, kway_ncut
from ncut_pytorch.ncut_pytorch import affinity_from_features, ncut

from my_ipadapter_model import load_ipadapter, image_grid, generate
from my_intrinsic_dim import get_intrinsic_dim
from dino_clip_featextract import extract_dino_image_embeds, extract_clip_image_embeds, dino_img_transform, img_transform_inv
from my_dino_correspondence import get_correspondence_plot, ncut_tsne_multiple_images, kway_cluster_per_image, kway_cluster_multiple_images, get_single_multi_discrete_rgbs, match_centers_three_images, match_centers_two_images, get_center_features
from compression_model_mkii import CompressionModel, train_compression_model, free_memory, get_fg_mask

plt.rcParams['font.family'] = 'monospace'

In [2]:
def load_images_helper(pil_images):
    if isinstance(pil_images[0], str):
        pil_images = [Image.open(image) for image in pil_images]
    # convert to RGB
    pil_images = [image.convert("RGB") for image in pil_images]
    return pil_images

image_paths = ["./images/jimi_portrait.jpg", "./images/jimi_action.jpg"]

images = load_images_helper(image_paths)
images_tensor = torch.stack([dino_img_transform(image) for image in images])
dino_image_embeds = extract_dino_image_embeds(images_tensor)[:, 1:, :]
b, l, c = dino_image_embeds.shape
h = w = int(l**0.5)
features = rearrange(dino_image_embeds, 'b l c -> (b l) c')

print(f'b: {b}, h: {h}, w: {w}')
print(features.shape)

Using cache found in /home/andrew/.cache/torch/hub/facebookresearch_dino_main


b: 2, h: 64, w: 64
torch.Size([8192, 384])


In [3]:
def differentiable_kway_ncut(features, n_segment):
    assert features.requires_grad == True
    A = affinity_from_features(features, distance='rbf', gamma=0.5)
    eigvec, eigval = ncut(A, n_segment)
    kway_eigvec = kway_ncut(eigvec)
    return eigvec, kway_eigvec

def differentiable_single_kway_ncut(features, n_segment):
    assert features.requires_grad == True

    features = rearrange(features, '(b l) c -> b l c', b=b)
    
    eigvec_list, kway_list = [], []
    for i in range(b):
        A = affinity_from_features(features[i], distance='rbf', gamma=0.5)
        eigv, eigval = ncut(A, n_segment)
        kway_eig = kway_ncut(eigv)
        eigvec_list.append(eigv)
        kway_list.append(kway_eig)
    
    eigvec = torch.cat(eigvec_list, dim=0)       # (b*l, c)
    kway_eigvec = torch.cat(kway_list, dim=0)    # (b*l, c)
    return eigvec, kway_eigvec

In [4]:
features = features.clone()
features.requires_grad_(True)
n_segment = 4
eigvec, kway_eigvec = differentiable_kway_ncut(features, n_segment=n_segment)
print(eigvec.requires_grad)

True


In [5]:
def channel_gradient_from_cluster(features, cluster_mask, kway_eigvec, cluster_idx):
    assert features.requires_grad == True
    if features.grad is not None:
        features.grad.zero_()
    loss = - kway_eigvec[cluster_mask, cluster_idx].abs().mean() 
    # maximize the absolute value of the eigvec
    loss.backward(retain_graph=True)
    grad = features.grad[cluster_mask]#.mean(0)  # C
    return grad

import matplotlib.pyplot as plt
def plot_channel_gradient(grad, cluster_mask, features, n_plot_images=None):
    n_images = len(images)
    n_plot_images = n_images if n_plot_images is None else n_plot_images
    
    cluster_mask = cluster_mask.reshape(n_images, h, w)
    features = features.reshape(n_images, h, w, -1)
    features = features.detach().cpu().numpy()
    
    grad_sorted = torch.argsort(grad, descending=False)
    
    fig, axes = plt.subplots(n_plot_images, 7, figsize=(14, 2*n_plot_images))
    for i in range(n_plot_images):
        axes[i, 0].imshow(images[i])
        axes[i, 1].imshow(cluster_mask[i])
        axes[i, 1].set_title(f"cluster mask")
        for top_k in range(5):
            ch_idx = grad_sorted[top_k]
            print(ch_idx)
            # axes[i, top_k+2].imshow(features[i, :, :, ch_idx], cmap='bwr', vmin=-1, vmax=1)
            print(features.shape)
            print(features[i, :, :, ch_idx].shape)
            axes[i, top_k+2].imshow(features[i, :, :, ch_idx], cmap='viridis')
            axes[i, top_k+2].set_title(f"top{top_k} ch {ch_idx}")
    for ax in axes.flatten():
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [6]:
for i in range(n_segment):
    cluster_mask = kway_eigvec.argmax(1) == i
    grad = channel_gradient_from_cluster(features, cluster_mask, kway_eigvec, i)
    # plt.hist(grad.detach().cpu().numpy(), bins=100)
    # plt.show()
    plot_channel_gradient(grad, cluster_mask, features)

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn